# RAGAS-style metrics: faithfulness, answer_relevancy, answer_correctness

Scores come from `llm-judge-evaluation.ipynb` (one deterministic judge call per
row, no RAGAS/embeddings), stored in `ragas_evaluation`. This notebook only
reads and aggregates them into mean-score tables along two axes, plus their
full combination:

1. **method x llm_reasoning_model x judge_model** -- the 3 retrieval methods
   (`agent`, `pgvector`, `neo4j`), each evaluated with a few different answer-
   generating reasoning models, and those answers scored by a judge model.
2. **question type** -- `relationship_discovery`, `cross_platform`,
   `impact_analysis` (plus whatever open-ended questions don't carry any of
   those flags, bucketed as `regular`).
3. **all of the above combined** -- method x reasoning model x judge model x
   question type in a single table.

`embedding_model` is dropped as an axis: every row in `ragas_evaluation` was
generated with `qwen06b`, so it carries no signal here. `context_precision` /
`context_recall` are always `NULL` (never computed by the judge notebook) and
are excluded too.

One known data wrinkle: `gemma-4-26b-a4b-it` was judged under two different
spellings of the same OpenRouter judge slug, `glm-5.3-flash` and
`z-ai/glm-5.3-flash`. Both are normalized to `glm-5.3-flash` below before
grouping, otherwise that reasoning model would be split into two artificially
small judge-model buckets.

In [1]:
import os
import re
from pathlib import Path

import pandas as pd
import psycopg2
from dotenv import load_dotenv

REPO_ROOT = Path.cwd().parent
load_dotenv(REPO_ROOT / ".env")

pd.set_option("display.max_rows", None)

METHOD_ORDER = ["agent", "pgvector", "neo4j"]
METRIC_COLUMNS = ["faithfulness", "answer_relevancy", "answer_correctness"]
CATEGORY_ORDER = ["regular", "relationship_discovery", "cross_platform", "impact_analysis"]
REASONING_MODEL_FILTER = "gpt-oss-120b"


def connect():
    conn_str = os.getenv("PG_CONNECTION_STRING")
    if conn_str:
        return psycopg2.connect(conn_str)
    return psycopg2.connect(host="localhost", dbname="docs", user="postgres", password="password")

## Load scores from `ragas_evaluation`, joined with `question` for the category flags

In [2]:
QUERY = """
    SELECT
        re.method,
        re.llm_reasoning_model,
        re.judge_model,
        re.faithfulness,
        re.answer_relevancy,
        re.answer_correctness,
        q.is_relationship_discovery,
        q.is_cross_platform,
        q.is_impact_analysis
    FROM ragas_evaluation re
    JOIN question q ON q.id = re.question_id
"""

with connect() as conn:
    df = pd.read_sql(QUERY, conn)

# Normalize the two spellings of the same OpenRouter judge slug.
df["judge_model"] = df["judge_model"].str.replace("z-ai/", "", regex=False)

print(f"{len(df)} scored rows")
df.head()

2399 scored rows


/var/folders/6s/4n20n38x5y571387phy8c5lc0000gn/T/ipykernel_50653/3297239710.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(QUERY, conn)


,method,llm_reasoning_model,judge_model,faithfulness,answer_relevancy,answer_correctness,is_relationship_discovery,is_cross_platform,is_impact_analysis
0,pgvector,gpt-oss-120b,glm-5.3-flash,0.80,0.75,0.65,False,False,False
1,agent,gpt-oss-120b,glm-5.3-flash,0.80,0.50,0.40,True,False,False
2,agent,gpt-oss-120b,glm-5.3-flash,0.75,0.65,0.55,False,False,False
3,agent,gpt-oss-120b,glm-5.3-flash,0.40,0.30,0.30,False,True,False
4,neo4j,gpt-oss-120b,glm-5.3-flash,0.70,0.60,0.50,False,False,False


## Mean scores by method x llm_reasoning_model x judge_model

In [3]:
mean_by_method_model_judge = (
    df.groupby(["llm_reasoning_model", "judge_model", "method"])[METRIC_COLUMNS]
    .mean()
    .reset_index()
)

mean_by_method_model_judge.to_csv("ragas_mean_scores_by_method_model_judge.csv", index=False)
print("Saved ragas_mean_scores_by_method_model_judge.csv")

(
    mean_by_method_model_judge
    .pivot(index=["llm_reasoning_model", "judge_model"], columns="method", values=METRIC_COLUMNS)
)

Saved ragas_mean_scores_by_method_model_judge.csv


faithfulness                    \
method                                      agent    neo4j pgvector   
llm_reasoning_model    judge_model                                    
deepseek-v4.1-flash    glm-5.3-flash      0.86165  0.84500  0.82800   
gemma-4-26b-a4b-it     glm-5.3-flash      0.73392  0.78575  0.83140   
gpt-oss-120b           glm-5.3-flash      0.72250  0.74675  0.76310   
llama-3.3-70b-instruct glm-5.3-flash      0.47900  0.60025  0.82375   

                                     answer_relevancy                    \
method                                          agent    neo4j pgvector   
llm_reasoning_model    judge_model                                        
deepseek-v4.1-flash    glm-5.3-flash         0.872000  0.85875  0.69550   
gemma-4-26b-a4b-it     glm-5.3-flash         0.578643  0.61650  0.63050   
gpt-oss-120b           glm-5.3-flash         0.681750  0.68650  0.66850   
llama-3.3-70b-instruct glm-5.3-flash         0.371500  0.48625  0.65825   

                                     answer_correctness                    
method                                            agent    neo4j pgvector  
llm_reasoning_model    judge_model                                         
deepseek-v4.1-flash    glm-5.3-flash           0.793000  0.77375  0.63115  
gemma-4-26b-a4b-it     glm-5.3-flash           0.521859  0.55275  0.60365  
gpt-oss-120b           glm-5.3-flash           0.591750  0.60150  0.60965  
llama-3.3-70b-instruct glm-5.3-flash           0.295000  0.41125  0.60050

## Mean scores by question type

Locked to `llm_reasoning_model = gpt-oss-120b` so the three methods are
compared under the same answer-generating model. Each row is bucketed into
exactly one of `relationship_discovery`, `cross_platform`, `impact_analysis`,
or `regular` (no flag set).

In [4]:
def categorize(row) -> str:
    if row["is_relationship_discovery"]:
        return "relationship_discovery"
    if row["is_cross_platform"]:
        return "cross_platform"
    if row["is_impact_analysis"]:
        return "impact_analysis"
    return "regular"


df["category"] = df.apply(categorize, axis=1)

category_df = df[df["llm_reasoning_model"] == REASONING_MODEL_FILTER]
print(category_df.groupby("category").size().reindex(CATEGORY_ORDER))

mean_by_category = (
    category_df.groupby(["category", "method"])[METRIC_COLUMNS]
    .mean()
    .reset_index()
)

mean_by_category.to_csv("ragas_mean_scores_by_question_category.csv", index=False)
print("Saved ragas_mean_scores_by_question_category.csv")

mean_by_category.pivot(index="category", columns="method", values=METRIC_COLUMNS).reindex(CATEGORY_ORDER)

category
regular                   150
relationship_discovery    150
cross_platform            150
impact_analysis           150
dtype: int64
Saved ragas_mean_scores_by_question_category.csv


faithfulness                 answer_relevancy         \
method                        agent  neo4j pgvector            agent  neo4j   
category                                                                      
regular                       0.845  0.790   0.8384            0.762  0.727   
relationship_discovery        0.709  0.681   0.8210            0.651  0.658   
cross_platform                0.736  0.799   0.5120            0.742  0.725   
impact_analysis               0.600  0.717   0.8810            0.572  0.636   

                                answer_correctness                  
method                 pgvector              agent  neo4j pgvector  
category                                                            
regular                   0.740              0.630  0.602   0.6336  
relationship_discovery    0.754              0.589  0.586   0.7170  
cross_platform            0.420              0.647  0.655   0.3860  
impact_analysis           0.760              0.501  0.563   0.7020

## All combinations: method x llm_reasoning_model x judge_model x question type

In [5]:
mean_by_all = (
    df.groupby(["llm_reasoning_model", "judge_model", "category", "method"])[METRIC_COLUMNS]
    .mean()
    .reset_index()
)

mean_by_all.to_csv("ragas_mean_scores_all_combinations.csv", index=False)
print("Saved ragas_mean_scores_all_combinations.csv")

mean_by_all.pivot(
    index=["llm_reasoning_model", "judge_model", "category"],
    columns="method",
    values=METRIC_COLUMNS,
).reindex(CATEGORY_ORDER, level="category")

Saved ragas_mean_scores_all_combinations.csv


faithfulness  \
method                                                             agent   
llm_reasoning_model    judge_model   category                              
deepseek-v4.1-flash    glm-5.3-flash regular                    0.912600   
                                     relationship_discovery     0.864000   
                                     cross_platform             0.877000   
                                     impact_analysis            0.793000   
gemma-4-26b-a4b-it     glm-5.3-flash regular                    0.778571   
                                     relationship_discovery     0.691000   
                                     cross_platform             0.728000   
                                     impact_analysis            0.739000   
gpt-oss-120b           glm-5.3-flash regular                    0.845000   
                                     relationship_discovery     0.709000   
                                     cross_platform             0.736000   
                                     impact_analysis            0.600000   
llama-3.3-70b-instruct glm-5.3-flash regular                    0.592000   
                                     relationship_discovery     0.344000   
                                     cross_platform             0.439000   
                                     impact_analysis            0.541000   

                                                                             \
method                                                       neo4j pgvector   
llm_reasoning_model    judge_model   category                                 
deepseek-v4.1-flash    glm-5.3-flash regular                 0.905   0.8670   
                                     relationship_discovery  0.828   0.8450   
                                     cross_platform          0.857   0.7310   
                                     impact_analysis         0.790   0.8690   
gemma-4-26b-a4b-it     glm-5.3-flash regular                 0.877   0.9140   
                                     relationship_discovery  0.714   0.9020   
                                     cross_platform          0.775   0.5836   
                                     impact_analysis         0.777   0.9260   
gpt-oss-120b           glm-5.3-flash regular                 0.790   0.8384   
                                     relationship_discovery  0.681   0.8210   
                                     cross_platform          0.799   0.5120   
                                     impact_analysis         0.717   0.8810   
llama-3.3-70b-instruct glm-5.3-flash regular                 0.746   0.8770   
                                     relationship_discovery  0.555   0.8990   
                                     cross_platform          0.562   0.6650   
                                     impact_analysis         0.538   0.8540   

                                                            answer_relevancy  \
method                                                                 agent   
llm_reasoning_model    judge_model   category                                  
deepseek-v4.1-flash    glm-5.3-flash regular                         0.89900   
                                     relationship_discovery          0.86600   
                                     cross_platform                  0.87900   
                                     impact_analysis                 0.84400   
gemma-4-26b-a4b-it     glm-5.3-flash regular                         0.62449   
                                     relationship_discovery          0.55200   
                                     cross_platform                  0.58300   
                                     impact_analysis                 0.55600   
gpt-oss-120b           glm-5.3-flash regular                         0.76200   
                                     relationship_discovery          0.65100   
                                     cross_platform                  0.74

## External judge runs: `output/requirement/ragas/*.csv`

Separate one-off judge runs (not written to `ragas_evaluation`), already
pre-aggregated as mean scores by method. Each file is named
`{judge}_llm_judge_mean_scores_by_method_{reasoning_model}.csv`; both are
parsed out of the filename and added as columns so every file can be
combined into one table.

In [6]:
EXTERNAL_RAGAS_DIR = REPO_ROOT / "output" / "requirement" / "ragas"
FILENAME_RE = re.compile(r"^(?P<judge>[^_]+)_llm_judge_mean_scores_by_method_(?P<reasoning_model>[^_]+)\.csv$")

external_frames = []
for path in sorted(EXTERNAL_RAGAS_DIR.glob("*.csv")):
    match = FILENAME_RE.match(path.name)
    if not match:
        print(f"Skipping unrecognized filename: {path.name}")
        continue
    frame = pd.read_csv(path)
    frame["judge_model"] = match.group("judge")
    frame["llm_reasoning_model"] = match.group("reasoning_model")
    external_frames.append(frame)

external_df = pd.concat(external_frames, ignore_index=True)
external_df = external_df[["llm_reasoning_model", "judge_model", "method"] + METRIC_COLUMNS]

print(f"{len(external_df)} rows from {len(external_frames)} files")
external_df.to_csv("ragas_external_judge_mean_scores.csv", index=False)
print("Saved ragas_external_judge_mean_scores.csv")

external_df.pivot(
    index=["llm_reasoning_model", "judge_model"],
    columns="method",
    values=METRIC_COLUMNS,
)

33 rows from 11 files
Saved ragas_external_judge_mean_scores.csv


faithfulness                     \
method                                 agent     neo4j pgvector   
llm_reasoning_model judge_model                                   
deepseek            glm             0.861650  0.845000  0.82800   
                    hy4             0.832450  0.825850  0.82165   
                    mimo            0.860050  0.850100  0.82925   
gemma               hy4             0.733317  0.779000  0.82245   
                    mimo            0.703920  0.758250  0.79410   
gptoss              glm             0.722500  0.746750  0.76310   
                    hy4             0.682100  0.706250  0.71570   
                    mimo            0.709400  0.731100  0.78785   
llama               glm             0.479000  0.600250  0.82375   
                    hy4             0.462900  0.581175  0.79760   
                    mimo            0.462000  0.583000  0.81260   

                                answer_relevancy                    \
method                                     agent    neo4j pgvector   
llm_reasoning_model judge_model                                      
deepseek            glm                 0.872000  0.85875  0.69550   
                    hy4                 0.814800  0.80320  0.64910   
                    mimo                0.865800  0.85280  0.71390   
gemma               hy4                 0.532613  0.58305  0.59865   
                    mimo                0.582513  0.62850  0.62005   
gptoss              glm                 0.681750  0.68650  0.66850   
                    hy4                 0.626450  0.63795  0.61915   
                    mimo                0.675900  0.68280  0.65990   
llama               glm                 0.371500  0.48625  0.65825   
                    hy4                 0.337750  0.44895  0.63210   
                    mimo                0.394150  0.49480  0.64695   

                                answer_correctness                    
method                                       agent    neo4j pgvector  
llm_reasoning_model judge_model                                       
deepseek            glm                   0.793000  0.77375  0.63115  
                    hy4                   0.728850  0.71955  0.59530  
                    mimo                  0.775750  0.76670  0.65940  
gemma               hy4                   0.475075  0.51755  0.55680  
                    mimo                  0.524523  0.56170  0.60340  
gptoss              glm                   0.591750  0.60150  0.60965  
                    hy4                   0.538100  0.54835  0.55735  
                    mimo                  0.582050  0.59610  0.61625  
llama               glm                   0.295000  0.41125  0.60050  
                    hy4                   0.278000  0.38325  0.57565  
                    mimo                  0.314250  0.41990  0.61035